In [2]:
from pathlib import Path
path = Path("C:\\Users\\ELGRM\\OneDrive\\Desktop\\Hands-On_Machine_Learning_with_Scikit-Learn-Keras-and-TensorFlow-2nd-Edition-Aurelien-Geron\\CHAPTER 14 Natural Language Processing with RNNs and Attention\\input.txt")
text = path.read_text(encoding="utf-8")



In [3]:
vocab = sorted(list(set(text.lower())))
print(vocab)


['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [4]:
char_to_id = {char: index for index, char in enumerate(vocab)}
id_to_char = {index : char for index, char in enumerate(vocab)}

In [5]:
id_to_char

{0: '\n',
 1: ' ',
 2: '!',
 3: '$',
 4: '&',
 5: "'",
 6: ',',
 7: '-',
 8: '.',
 9: '3',
 10: ':',
 11: ';',
 12: '?',
 13: 'a',
 14: 'b',
 15: 'c',
 16: 'd',
 17: 'e',
 18: 'f',
 19: 'g',
 20: 'h',
 21: 'i',
 22: 'j',
 23: 'k',
 24: 'l',
 25: 'm',
 26: 'n',
 27: 'o',
 28: 'p',
 29: 'q',
 30: 'r',
 31: 's',
 32: 't',
 33: 'u',
 34: 'v',
 35: 'w',
 36: 'x',
 37: 'y',
 38: 'z'}

In [6]:
import torch 

def encode_text(text):
    return torch.tensor([char_to_id[char] for char in text.lower()], dtype=torch.long)

def decode_text(char_ids):
    return "".join([id_to_char[char_id.item()] for char_id in char_ids])



In [7]:
from torch.utils.data import Dataset , DataLoader

class CharDataset(Dataset):
    def __init__(self , text , window_length):
        self.encoded_text = encode_text(text)
        self.window_length = window_length

    def __len__(self):
        return len(self.encoded_text) - self.window_length

    def __getitem__(self,idx):
        if idx >= len(self):
            raise IndexError("dataset is out of range")
        else:
            end = idx + self.window_length
            window = self.encoded_text[idx : end]
            target = self.encoded_text[idx+1 : end+1]
            return window , target

In [8]:
window_length = 50 
batch_size = 512
train_set = CharDataset(text[:1_000_000], window_length)
valid_set = CharDataset(text[1_000_000:1_060_000], window_length)
test_set = CharDataset(text[1_060_000:], window_length)
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(valid_set, batch_size=batch_size)
test_loader = DataLoader(test_set, batch_size=batch_size)



In [9]:
from torch import nn 

class ShakespeareModel(nn.Module):
    def __init__(self, vocab_size , n_layer =2 , embed_dim = 10 , hidden_dim = 128 , dropout = 0.1):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.gru = nn.GRU(embed_dim , hidden_dim , n_layer ,batch_first = True , dropout = dropout)
        self.output = nn.Linear(hidden_dim , vocab_size)

    def forward(self , x):
        embeddings = self.embed(x)
        outputs ,_states = self.gru(embeddings)
        outputs = self.output(outputs).permute(0, 2, 1)
        return outputs

torch.manual_seed(42)
model = ShakespeareModel(len(vocab))

       

In [10]:
import torch
import torch.nn as nn
from tqdm.auto import tqdm

# ─────────────────────────────────────────────────────────────
# 1. Device, Loss Function & Optimizer
# ─────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model.to(device)

# CrossEntropyLoss expects:
#   Inputs:  (batch_size, num_classes, seq_len)
#   Targets: (batch_size, seq_len)
loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# ─────────────────────────────────────────────────────────────
# 2. Training Loop
# ─────────────────────────────────────────────────────────────
num_epochs = 10

for epoch in range(1, num_epochs + 1):
    # ── Training Phase ──
    model.train()
    train_loss = 0.0
    correct_tokens = 0
    total_tokens = 0

    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch:02d}/{num_epochs} [Train]")
    for X_batch, y_batch in train_pbar:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        # 1. Forward pass
        logits = model(X_batch)  # shape: (batch, vocab_size=39, seq_len)

        # 2. Compute loss
        loss = loss_fn(logits, y_batch)

        # 3. Backward pass & optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        # Compute accuracy across characters
        # Argmax along dimension 1 (vocab_size)
        preds = logits.argmax(dim=1)  # shape: (batch, seq_len)
        correct_tokens += (preds == y_batch).sum().item()
        total_tokens += y_batch.numel()

        train_pbar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "acc": f"{(correct_tokens / total_tokens):.3f}"
        })

    avg_train_loss = train_loss / len(train_loader)
    train_acc = correct_tokens / total_tokens

    # ── Validation Phase ──
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for X_batch, y_batch in valid_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            logits = model(X_batch)
            loss = loss_fn(logits, y_batch)

            val_loss += loss.item()
            preds = logits.argmax(dim=1)
            val_correct += (preds == y_batch).sum().item()
            val_total += y_batch.numel()

    avg_val_loss = val_loss / len(valid_loader)
    val_acc = val_correct / val_total

    print(f"\nEpoch {epoch:02d} Summary: "
          f"Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.3f} || "
          f"Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.3f}\n")



c:\Users\ELGRM\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu


Epoch 01/10 [Train]:   1%|          | 11/1954 [00:09<28:46,  1.13it/s, loss=3.2509, acc=0.133]


KeyboardInterrupt: 

In [11]:
model.eval()
text = "To be or not to b"
encoded_text = encode_text(text).unsqueeze(dim=0)
with torch.no_grad():
    Y_logits = model(encoded_text)
    predicted_char_id = Y_logits[0, :, -1].argmax().item()
    predicted_char = id_to_char[predicted_char_id]
    print(predicted_char)


In [12]:
import torch.nn.functional as F

def next_char(model, text, temperature=1):
    encoded_text = encode_text(text).unsqueeze(dim=0).to(device)
    with torch.no_grad():
        Y_logits = model(encoded_text)
        Y_probas = F.softmax(Y_logits[0, :, -1] / temperature, dim=-1)
        predicted_char_id = torch.multinomial(Y_probas, num_samples=1).item()
    return id_to_char[predicted_char_id]

In [13]:
def extend_text(model, text, n_chars=80, temperature=1):
    for _ in range(n_chars):
        text += next_char(model, text, temperature)
    return text

In [14]:
print(extend_text(model, "To be or not to b", temperature=0.01))

To be or not to b                                                                                


In [15]:
print(extend_text(model, "To be or not to b", temperature=0.4))


To be or not to b           
         
 te e     s                               a             ne


In [16]:
from datasets import load_dataset

imdb_dataset  = load_dataset("stanfordnlp/imdb")


In [17]:
split = imdb_dataset["train"].train_test_split(train_size=0.8 ,seed = 42)
imdb_train_set , imdb_valid_set = split["train"], split["test"]
imdb_test_set = imdb_dataset["test"]



In [18]:
import tokenizers


bpe_model = tokenizers.models.BPE(unk_token="<unk>")
bpe_tokenizer = tokenizers.Tokenizer(bpe_model)
bpe_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Whitespace()
special_tokens = ["<pad>", "<unk>"]
bpe_trainer = tokenizers.trainers.BpeTrainer(vocab_size=1000,
special_tokens=special_tokens)
train_reviews = [review["text"].lower() for review in imdb_train_set]
bpe_tokenizer.train_from_iterator(train_reviews, bpe_trainer)

In [ ]:
bpe_tokenizer.encode_batch(train_reviews[:3])

[Encoding(num_tokens=281, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing]),
 Encoding(num_tokens=114, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing]),
 Encoding(num_tokens=285, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])]

In [20]:
import transformers 

gpt2_tokenizer = transformers.AutoTokenizer.from_pretrained("gpt2")
gpt2_encoding = gpt2_tokenizer(train_reviews[:3] , tranculate=True , max_length =500 )




In [22]:
gpt2_token_ids = gpt2_encoding["input_ids"][0][:10]
gpt2_token_ids

[14247, 35030, 1690, 423, 257, 1688, 8046, 13, 484, 1690]

In [23]:
gpt2_tokenizer.decode(gpt2_token_ids)

'stage adaptations often have a major fault. they often'

In [ ]:
bert_tokenizer = transformers.AutoTokenizer.from_pretrained("bert-base-uncased")
bert_encoding = bert_tokenizer(train_reviews[:3], padding=True,
truncation=True, max_length=500,
return_tensors="pt")

c:\Users\ELGRM\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ELGRM\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [27]:
albert_tokenizer = transformers.AutoTokenizer.from_pretrained("albert-base-v2")
albert_encoding = albert_tokenizer(train_reviews[:3], padding=True ,truncation=True, max_length=500)

In [29]:
def collate_fn(batch , tokenizer=bert_tokenizer):
    reviews  = [review["text"] for  review in batch]
    labels = [[review["label"]] for review in batch]

    encoding = tokenizer(reviews , padding = True , trancation = True , max_length = 200 , return_tensors = "pt")
    labels = torch.tensor(labels, dtype=torch.float32)
    return encoding , labels


In [30]:
batch_size = 256

imdb_train_loader = DataLoader(imdb_train_set , batch_size=batch_size , shuffle=True , collate_fn=collate_fn)
imdb_valid_loader = DataLoader(imdb_valid_set , batch_size=batch_size , shuffle=True , collate_fn=collate_fn)
imdb_test_loader = DataLoader(imdb_test_set , batch_size=batch_size , shuffle=True , collate_fn=collate_fn)


In [ ]:
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

class SentimentAnalysisModel(nn.Module):
    def __init__(self, vocab_size, n_layers=2, embed_dim=128, hidden_dim=64,
                    pad_id=0, dropout=0.2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim,
        padding_idx=pad_id)
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers, bidirectional=True , 
        batch_first=True, dropout=dropout)
        self.output = nn.Linear(2*hidden_dim, 1)
    def forward(self, encodings):
        embeddings = self.embed(encodings["input_ids"])
        lengths = encodings["attention_mask"].sum(dim=1)
        packed = pack_padded_sequence(embeddings, lengths=lengths.cpu(),
        batch_first=True, enforce_sorted=False)
        _outputs, hidden_states = self.gru(packed)
        n_dims = self.output.in_features
        top_states = hidden_states[-2:].permute(1, 0, 2).reshape(-1, n_dims)
        return self.output(top_states)

In [33]:
bert_model = transformers.AutoModel.from_pretrained("bert-base-uncased")
bert_model.embeddings.word_embeddings

c:\Users\ELGRM\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ELGRM\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 739.50it/s]
[transformers] BertModel LOAD REPORT from: bert-base-un

Embedding(30522, 768, padding_idx=0)

In [34]:
from transformers import pipeline
model_name = "distilbert-base-uncased-finetuned-sst-2-english"
classifier_imdb = pipeline("sentiment-analysis", model=model_name,
                            truncation=True, max_length=512)

classifier_imdb(train_reviews[:10])

c:\Users\ELGRM\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ELGRM\.cache\huggingface\hub\models--distilbert-base-uncased-finetuned-sst-2-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 104/104 [00:00<00:00, 914.97it/s]

[{'label': 'POSITIVE', 'score': 0.9996108412742615},
 {'label': 'POSITIVE', 'score': 0.9998623132705688},
 {'label': 'NEGATIVE', 'score': 0.9943684935569763},
 {'label': 'POSITIVE', 'score': 0.997913658618927},
 {'label': 'POSITIVE', 'score': 0.999544084072113},
 {'label': 'NEGATIVE', 'score': 0.9845331311225891},
 {'label': 'POSITIVE', 'score': 0.9859278202056885},
 {'label': 'POSITIVE', 'score': 0.9993758797645569},
 {'label': 'POSITIVE', 'score': 0.9978922009468079},
 {'label': 'NEGATIVE', 'score': 0.9997020363807678}]

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import DataLoader
from datasets import load_dataset
import tokenizers

In [5]:
nmt_original_valid_set , nmt_test_set = load_dataset(
    path="ageron/tatoeba_mt_train", name= "eng-spa",
    split=["validation" , "test"])

split = nmt_original_valid_set.train_test_split(train_size = 0.8 ,seed=42)
nmt_train_set , nmt_valid_set = split["train"] ,split["test"]

In [10]:
nmt_train_set[0]

{'source_text': 'Tom tried to break up the fight.',
 'target_text': 'Tom trató de disolver la pelea.',
 'source_lang': 'eng',
 'target_lang': 'spa'}

In [12]:
def train_eng_spa():
    for pair in nmt_train_set:
        yield pair["source_text"]
        yield pair["target_text"]

max_length = 256
vocab_size = 10_000
nmt_tokenizer_model = tokenizers.models.BPE(unk_token="<unk>")
nmt_tokenizer = tokenizers.Tokenizer(nmt_tokenizer_model)
nmt_tokenizer.enable_padding(pad_id = 0 , pad_token="<pad>")
nmt_tokenizer.enable_truncation(max_length=max_length)
nmt_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Whitespace()
nmt_tokenizer_trainer  =tokenizers.trainers.BpeTrainer(vocab_size=vocab_size , special_tokens=["<pad>", "<unk>", "<s>", "</s>"])
nmt_tokenizer.train_from_iterator(train_eng_spa() , nmt_tokenizer_trainer)



In [13]:
nmt_tokenizer.encode("I like soccer").ids

[43, 401, 4381]

In [14]:
nmt_tokenizer.encode("<s> Me gusta el fútbol").ids

[2, 396, 582, 219, 3356]

In [15]:
from collections import namedtuple
fields = ["src_token_ids", "src_mask", "tgt_token_ids", "tgt_mask"]
class NmtPair(namedtuple("NmtPairBase", fields)):
    def to(self, device):
        return NmtPair(self.src_token_ids.to(device), self.src_mask.to(device),
                self.tgt_token_ids.to(device), self.tgt_mask.to(device))

In [26]:
def nmt_collate_fn(batch):
    src_texts = [pair["source_text"] for pair in batch]
    tgt_texts = [f"<s> {pair['target_text']} </s>" for pair in batch]
    src_encodings = nmt_tokenizer.encode_batch(src_texts)
    tgt_encodings = nmt_tokenizer.encode_batch(tgt_texts)
    src_token_ids = torch.tensor([enc.ids for enc in src_encodings])
    tgt_token_ids = torch.tensor([enc.ids for enc in tgt_encodings])
    src_mask = torch.tensor([enc.attention_mask for enc in src_encodings])
    tgt_mask = torch.tensor([enc.attention_mask for enc in tgt_encodings])
    inputs = NmtPair(src_token_ids ,src_mask ,tgt_token_ids[: ,:-1] , tgt_mask[:, :-1])
    labels = tgt_token_ids[: ,1:]
    return inputs , labels


In [32]:
batch_size =32

nmt_train_loader = DataLoader(nmt_train_set , batch_size=batch_size , collate_fn = nmt_collate_fn ,shuffle = True)
nmt_valid_loader = DataLoader(nmt_valid_set , batch_size=batch_size , collate_fn = nmt_collate_fn )
nmt_test_loader = DataLoader(nmt_test_set , batch_size=batch_size , collate_fn = nmt_collate_fn )


In [28]:
class NmtModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=512, pad_id=0, hidden_dim=512,
                    n_layers=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.encoder = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers,
                            batch_first=True)
        self.decoder = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers,
                            batch_first=True)
        self.output = nn.Linear(hidden_dim, vocab_size)
    def forward(self, pair):
        src_embeddings = self.embed(pair.src_token_ids)
        tgt_embeddings = self.embed(pair.tgt_token_ids)
        src_lengths = pair.src_mask.sum(dim=1)
        src_packed = pack_padded_sequence(
                src_embeddings, lengths=src_lengths.cpu(),
                batch_first=True, enforce_sorted=False)
        _, hidden_states = self.encoder(src_packed)
        outputs, _ = self.decoder(tgt_embeddings, hidden_states)
        return self.output(outputs).permute(0, 2, 1)
    
torch.manual_seed(42)
vocab_size = nmt_tokenizer.get_vocab_size()
nmt_model = NmtModel(vocab_size)

In [ ]:
import torch
import torch.nn as nn
from tqdm.auto import tqdm

# ─────────────────────────────────────────────────────────────
# 1. Device, Loss Function & Optimizer
# ─────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

nmt_model.to(device)

# CrossEntropyLoss expects:
#   Inputs:  (batch_size, num_classes, seq_len)
#   Targets: (batch_size, seq_len)
loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(nmt_model.parameters(), lr=1e-3)

# ─────────────────────────────────────────────────────────────
# 2. Training Loop
# ─────────────────────────────────────────────────────────────
num_epochs = 10

for epoch in range(1, num_epochs + 1):
    # ── Training Phase ──
    nmt_model.train()
    train_loss = 0.0
    correct_tokens = 0
    total_tokens = 0

    train_pbar = tqdm(nmt_train_loader, desc=f"Epoch {epoch:02d}/{num_epochs} [Train]")
    for X_batch, y_batch in train_pbar:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        # 1. Forward pass
        logits = nmt_model(X_batch)  # shape: (batch, vocab_size=39, seq_len)

        # 2. Compute loss
        loss = loss_fn(logits, y_batch)

        # 3. Backward pass & optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        # Compute accuracy across characters
        # Argmax along dimension 1 (vocab_size)
        preds = logits.argmax(dim=1)  # shape: (batch, seq_len)
        correct_tokens += (preds == y_batch).sum().item()
        total_tokens += y_batch.numel()

        train_pbar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "acc": f"{(correct_tokens / total_tokens):.3f}"
        })

    avg_train_loss = train_loss / len(train_loader)
    train_acc = correct_tokens / total_tokens

    # ── Validation Phase ──
    nmt_model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for X_batch, y_batch in nmt_valid_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            logits = model(X_batch)
            loss = loss_fn(logits, y_batch)

            val_loss += loss.item()
            preds = logits.argmax(dim=1)
            val_correct += (preds == y_batch).sum().item()
            val_total += y_batch.numel()

    avg_val_loss = val_loss / len(valid_loader)
    val_acc = val_correct / val_total

    print(f"\nEpoch {epoch:02d} Summary: "
          f"Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.3f} || "
          f"Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.3f}\n")



Using device: cpu


Epoch 01/10 [Train]:   6%|▌         | 295/4933 [02:35<40:49,  1.89it/s, loss=2.6625, acc=0.729]  


KeyboardInterrupt: 

In [45]:
import torch

def train(model, train_loader, valid_loader, loss_fn, optimizer, metric=None, n_epochs=5, device="cuda" if torch.cuda.is_available() else "cpu"):
    


    for epoch in range(1, n_epochs + 1):
        # ─────────────────────────────────────────────────────────────
        # 1. Training Phase
        # ─────────────────────────────────────────────────────────────
        model.train()
        total_train_loss = 0.0
        if metric: 
            metric.reset()

        for batch in train_loader:
            if isinstance(batch, (tuple, list)):
                inputs, targets = batch
            else:
                inputs, targets = batch.inputs, batch.labels

            inputs = inputs.to(device) if hasattr(inputs, "to") else inputs
            targets = targets.to(device)

            # Forward & backward
            outputs = model(inputs)
            loss = loss_fn(outputs, targets)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()
            if metric: 
                metric.update(outputs, targets)

        avg_train_loss = total_train_loss / len(train_loader)
        train_score = metric.compute().item() if metric else 0.0

        # ─────────────────────────────────────────────────────────────
        # 2. Validation Phase
        # ─────────────────────────────────────────────────────────────
        model.eval()
        total_val_loss = 0.0
        if metric: 
            metric.reset()

        with torch.no_grad():
            for batch in valid_loader:
                if isinstance(batch, (tuple, list)):
                    inputs, targets = batch
                else:
                    inputs, targets = batch.inputs, batch.labels

                inputs = inputs.to(device) if hasattr(inputs, "to") else inputs
                targets = targets.to(device)

                outputs = model(inputs)
                val_loss = loss_fn(outputs, targets)

                total_val_loss += val_loss.item()
                if metric: 
                    metric.update(outputs, targets)

        avg_val_loss = total_val_loss / len(valid_loader)
        val_score = metric.compute().item() if metric else 0.0

        # Print summary for this epoch
        print(f"Epoch {epoch:02d}/{n_epochs:02d} | "
              f"Train Loss: {avg_train_loss:.4f} - Metric: {train_score:.4f} | "
              f"Val Loss: {avg_val_loss:.4f} - Metric: {val_score:.4f}")

In [47]:
import torch
import torch.nn as nn
import torchmetrics

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
n_epochs = 10

xentropy = nn.CrossEntropyLoss(ignore_index=0)  # ignore <pad> tokens
optimizer = torch.optim.NAdam(nmt_model.parameters(), lr=0.001)

accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=vocab_size)
accuracy = accuracy.to(device)

# Pass arguments using keywords so the order can never be mixed up again:
history = train(
    model=nmt_model,
    train_loader=nmt_train_loader,
    valid_loader=nmt_valid_loader,
    loss_fn=xentropy,
    optimizer=optimizer,
    metric=accuracy,
    n_epochs=n_epochs,
    device=device
)

KeyboardInterrupt: 

In [ ]:
%pip install torchmetrics